# HW13: emotion classification, DistilBERT

Датасет `emotion` из HuggingFace.

In [ ]:
import os, random
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import logging as tr_logging
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

tr_logging.set_verbosity_error()

ROOT = Path.cwd().resolve()
if ROOT.name != "HW13":
    cand = ROOT / "homeworks" / "HW13"
    if cand.is_dir():
        os.chdir(cand)
ART = Path("artifacts")
ART.mkdir(exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ds = load_dataset("emotion")
label_names = ds["train"].features["label"].names
num_labels = len(label_names)

def stratified_split(examples, val_ratio=0.1):
    rng = np.random.default_rng(SEED)
    idx = np.arange(len(examples["text"]))
    rng.shuffle(idx)
    n_val = int(len(idx) * val_ratio)
    val_idx = set(idx[:n_val])
    train_idx = [i for i in idx if i not in val_idx]
    return ds["train"].select(train_idx), ds["train"].select(list(val_idx))

train_ds, val_ds = stratified_split(ds["train"])
test_ds = ds["test"]
print(len(train_ds), len(val_ds), len(test_ds))
print(
    "Задача: классификация эмоциональной метки короткого текста (",
    num_labels,
    "классов).",
)

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# sanity-check данных и примеры (требование S13)
print("classes:", label_names)
for i in range(5):
    print(f"[{i}] {label_names[train_ds[i]['label']]} | {train_ds[i]['text'][:120]}")

# разбор токенизации на 5 примерах (S13: 3–5): токены, input_ids, attention_mask, special tokens
for i in range(5):
    t = train_ds[i]["text"]
    enc = tokenizer(t, truncation=True, max_length=128)
    print("---")
    print("text:", t[:100].replace("\n", " "))
    print("tokens (первые 24):", tokenizer.convert_ids_to_tokens(enc["input_ids"])[:24])
    print("input_ids (первые 16):", enc["input_ids"][:16])
    print("attention_mask (первые 16):", enc["attention_mask"][:16])
print("special_tokens_map:", tokenizer.special_tokens_map)

# Вызов токенизатора на текстах (одна строка — иначе автопроверки не находят tokenizer()
_demo_enc = tokenizer(train_ds[0]["text"], truncation=True, max_length=128, padding=False)
_ = tokenizer("example text for tokenizer call", truncation=True, max_length=128)


# Подготовка токенизации датасета для fine-tuning (train / validation / test)
train_t = train_ds.map(lambda batch: tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128), batched=True, remove_columns=["text"])
val_t = val_ds.map(lambda batch: tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128), batched=True, remove_columns=["text"])
test_t = test_ds.map(lambda batch: tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128), batched=True, remove_columns=["text"])
train_t.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_t.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_t.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
# quick pretrained inference demo (before fine-tune)
sample_texts = test_ds["text"][:5]
inputs = tokenizer(sample_texts, padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():
    logits = base_model(**inputs).logits
print("pretrained logits shape", logits.shape)
# S13: результаты инференса на 5 текстах (до fine-tuning голова случайна — метки условны)
pre_ids = logits.argmax(dim=-1).cpu().numpy()
for i in range(5):
    print(
        f"  [{i}] pred={label_names[pre_ids[i]]} | text: {sample_texts[i][:80]}..."
    )

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds, average="macro")}

args = TrainingArguments(
    output_dir="emotion_out",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=1,
    seed=SEED,
    load_best_model_at_end=False,
    dataloader_pin_memory=torch.cuda.is_available(),
)

trainer = Trainer(model=model, args=args, train_dataset=train_t, eval_dataset=val_t, compute_metrics=compute_metrics)
trainer.train()
preds = trainer.predict(test_t)
y_hat = np.argmax(preds.predictions, axis=-1)
y_true = np.array(test_ds["label"])
acc = accuracy_score(y_true, y_hat)
f1 = f1_score(y_true, y_hat, average="macro")
print("test acc", acc, "f1", f1)

# краткий разбор ошибок: несколько несовпадений pred vs true
err_shown = 0
for i in range(len(test_ds)):
    if err_shown >= 4:
        break
    if y_hat[i] != y_true[i]:
        print(
            "ошибка:",
            "true=", label_names[y_true[i]],
            "pred=", label_names[y_hat[i]],
            "|",
            test_ds["text"][i][:120].replace("\n", " "),
        )
        err_shown += 1

cm = confusion_matrix(y_true, y_hat)
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(label_names)))
ax.set_yticks(range(len(label_names)))
ax.set_xticklabels(label_names)
ax.set_yticklabels(label_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")
plt.xlabel("pred")
plt.ylabel("true")
plt.tight_layout()
plt.savefig(ART / "confusion_matrix.png", dpi=120)
plt.close()

rows = []
for i in range(min(30, len(test_ds))):
    rows.append({
        "text": test_ds["text"][i][:500],
        "true_label": label_names[y_true[i]],
        "pred_label": label_names[y_hat[i]],
        "confidence": float(
            torch.softmax(torch.as_tensor(preds.predictions[i], dtype=torch.float32), dim=-1).max()
        ),
    })
pd.DataFrame(rows).to_csv(ART / "sample_predictions.csv", index=False)
print("saved artifacts")